In [7]:
# Imports y configuración

import pandas as pd
import numpy as np
from pathlib import Path

import warnings
warnings.filterwarnings("ignore")

DATA_DIR = Path("../../data/dashboard")
VAL_DIR = DATA_DIR / "validation"
VAL_DIR.mkdir(parents=True, exist_ok=True)

RISK_FREE_ANNUAL = 0.03   # 3% anual
TRADING_DAYS = 252
rf_daily = (1 + RISK_FREE_ANNUAL)**(1/TRADING_DAYS) - 1


In [10]:
# Cargar señales del modelo

signals_path = DATA_DIR / "model_results" / "signals_raw.parquet"
signals_all = pd.read_parquet(signals_path, engine="fastparquet")

signals_all = signals_all.rename(columns={
    "Date": "date",
    "Ticker": "ticker",
    # descomenta una de estas según corresponda:
    # "Close": "close",
    # "Adj Close": "close",
})

signals_all["date"] = pd.to_datetime(signals_all["date"])

signals_all.head()



,ticker,date,close,y_true,y_proba,signal
0,UNH,2015-01-02 00:00:00-05:00,85.048462,0,0.531321,1
1,UNH,2015-01-05 00:00:00-05:00,83.647598,0,0.535999,1
2,UNH,2015-01-06 00:00:00-05:00,83.478821,1,0.536521,1
3,UNH,2015-01-07 00:00:00-05:00,84.331169,1,0.533554,1
4,UNH,2015-01-08 00:00:00-05:00,88.356575,0,0.523683,1


In [11]:
# Función de Backtesting Completa

def compute_backtesting(df_ticker: pd.DataFrame):
    df = df_ticker.sort_values("date").copy()

    df["ret_benchmark"] = df["close"].pct_change().fillna(0)

    # Usamos señal del día anterior
    df["position"] = df["signal"].shift(1).fillna(0)
    df["ret_strategy"] = df["position"] * df["ret_benchmark"]

    # Curvas acumuladas
    df["cum_strategy"] = (1 + df["ret_strategy"]).cumprod()
    df["cum_benchmark"] = (1 + df["ret_benchmark"]).cumprod()

    # Métricas
    excess = df["ret_strategy"] - rf_daily
    sharpe = np.sqrt(TRADING_DAYS) * excess.mean() / excess.std() if excess.std() > 0 else np.nan
    
    negative_excess = excess[excess < 0]
    downside_std = negative_excess.std()
    sortino = (np.sqrt(TRADING_DAYS) * excess.mean() / downside_std 
               if downside_std and not np.isnan(downside_std) else np.nan)
    
    max_dd_strategy = (df["cum_strategy"] / df["cum_strategy"].cummax() - 1).min()
    max_dd_benchmark = (df["cum_benchmark"] / df["cum_benchmark"].cummax() - 1).min()

    metrics = pd.DataFrame({
        "metric": [
            "Sharpe",
            "Sortino",
            "Retorno total estrategia",
            "Retorno total buy & hold",
            "Max drawdown estrategia",
            "Max drawdown buy & hold"
        ],
        "strategy": [
            sharpe,
            sortino,
            df["cum_strategy"].iloc[-1] - 1,
            df["cum_benchmark"].iloc[-1] - 1,
            max_dd_strategy,
            max_dd_benchmark
        ]
    })

    return df, metrics


In [12]:
# Loop para todos los tickers

for ticker, df_t in signals_all.groupby("ticker"):
    print("Procesando:", ticker)

    bt_df, metrics_df = compute_backtesting(df_t)

    # Guardar curvas
    bt_df[["date", "cum_strategy", "cum_benchmark"]].to_csv(
        VAL_DIR / f"backtest_{ticker}.csv", 
        index=False
    )

    # Guardar métricas
    metrics_df.to_csv(
        VAL_DIR / f"metrics_{ticker}.csv", 
        index=False
    )


Procesando: UNH
Procesando: XOM


In [14]:
# Guardar señales unificadas

signals_all.to_parquet(
    VAL_DIR / "signals_all_tickers.parquet",
    index=False,
    engine="fastparquet"
)
signals_all.shape


(5032, 6)

In [15]:
# Guardar matriz de correlación

corr = signals_all.drop(columns=["date","ticker"]).corr()
corr.to_csv(DATA_DIR / "model_results" / "features_corr.csv")
corr.shape


(4, 4)

In [16]:
# ROC curve y reporte

from sklearn.metrics import roc_curve, classification_report

roc_list = []
metrics_list = []

for ticker, df_t in signals_all.groupby("ticker"):

    y_true = df_t["y_true"]
    y_proba = df_t["y_proba"]
    y_pred = (df_t["y_proba"] >= 0.5).astype(int)

    fpr, tpr, _ = roc_curve(y_true, y_proba)
    df_roc = pd.DataFrame({"ticker": ticker, "fpr": fpr, "tpr": tpr})
    roc_list.append(df_roc)

    report = classification_report(y_true, y_pred, output_dict=True)
    rep_df = pd.DataFrame(report).T
    rep_df["ticker"] = ticker
    metrics_list.append(rep_df)

roc_df = pd.concat(roc_list)
metrics_df = pd.concat(metrics_list)

roc_df.to_csv(DATA_DIR / "model_results" / "roc_curve.csv", index=False)
metrics_df.to_csv(DATA_DIR / "model_results" / "classification_metrics.csv")
